In [ ]:
!pip install pyspark
from pyspark import SparkConf, SparkContext
conf = SparkConf().setAppName("Prova_esame")
sc = SparkContext(conf=conf)

In [11]:
outputPath1 = "./output1/"
purchasesRDD = sc.textFile("./data/Purchases.txt")

#Task 1

In [12]:
def cleanFields(row):
  fields=row.split(",")
  year=fields[0].split("/")[0]
  id=fields[1]
  return (id, year)

def countYears(purchasesYears):
  year2022=0
  year2023=0
  for year in purchasesYears:
    if year == '2022':
      year2022 += 1
    elif year == '2023':
      year2023 += 1

  return (year2022,year2023)

cleanPurchasesRDD=purchasesRDD.map(cleanFields)
groupedIdsRDD = cleanPurchasesRDD.groupByKey().mapValues(countYears)
max2022=groupedIdsRDD.map(lambda row: row[1][0]).max()
max2023=groupedIdsRDD.map(lambda row: row[1][1]).max()
filteredRDD=groupedIdsRDD.filter(lambda x: x[1][0]==max2022 or x[1][1]==max2023)
outputRDD=filteredRDD.map(lambda x: x[0])
outputRDD.saveAsTextFile(outputPath1)

#Task 2

In [34]:
outputPath2="./output2/"
catalogueRDD=sc.textFile("./data/Catalogue.txt")

In [77]:
def newCleanFields(row):
  fields=row.split(",")
  userId=fields[1]
  prodId=fields[2]
  year=fields[0].split("/")[0]

  return (prodId, (userId,year))

def cleanCatalogue(row):
  fields=row.split(",")
  prodId=fields[0]
  category=fields[2]
  return (prodId, category)

def listCategories(row):
  fields=row.split(",")
  category=fields[2]
  return category


categoriesRDD=catalogueRDD.map(listCategories).distinct()
newCleanedRDD=purchasesRDD.map(newCleanFields).filter(lambda x: x[1][1]=='2022' or x[1][1]=='2023').mapValues(lambda val: val[0]).distinct().mapValues(lambda val: 1).reduceByKey(lambda v1,v2: v1+v2)
cleanedCatalogueRDD=catalogueRDD.map(cleanCatalogue)
joinedRDD= newCleanedRDD.join(cleanedCatalogueRDD).map(lambda val: (val[1][1],(val[0],val[1][0]))).cache()

maxPerCategoryRDD=joinedRDD.map(lambda val: (val[0],val[1][1])).reduceByKey(lambda a,b: max(a,b))
selectedItemsRDD = joinedRDD.join(maxPerCategoryRDD).filter(lambda val: val[1][0][1]==val[1][1]).map(lambda x: (x[0],x[1][0][0])).cache()

categoriesWithPurchasesRDD=selectedItemsRDD.map(lambda x: x[0]).distinct()
missingCategoriresRDD=categoriesRDD.subtract(categoriesWithPurchasesRDD).map(lambda val: (val, "No Purhcases"))
finalRDD=selectedItemsRDD.union(missingCategoriresRDD)

finalRDD.saveAsTextFile(outputPath2)
